In [2]:
import os
import glob
pattern = os.path.join("..", "data", "query-results-raw-*.json")
files = glob.glob(pattern)

if not files:
    print("Error: No data files found matching the pattern.")
    raise ValueError("No matching data files found.")


Error: No data files found matching the pattern.


ValueError: No matching data files found.

In [18]:
from typing import List
import pandas as pd
import json
import numpy as np
#TODO: Here instead of this we should calculate hit rate with relation to depth of session within sequence (so switching resets depth)
#TODO: In addition look at hit rate in general with relation to sequence depth
def calculate_session_hit_rates(filenames: List[str]) -> pd.DataFrame:
    """
    Parses result files to calculate average hit rates based on session context:
    new session, existing session, or within the current session.
    """
    records = []

    for path in sorted(filenames):
        algo_label = os.path.basename(path).replace("query-results-raw-", "").replace(".json", "")

        with open(path, 'r') as file:
            data = json.load(file)

        seen_sessions = set()
        prev_session_id = None

        for entry in data:
            seq_element = entry.get("sequenceElement", {})
            session_info = seq_element.get("session", {})

            if "sessionId" not in session_info:
                continue

            current_session_id = session_info["sessionId"]
            template = seq_element.get("template")

            cache_str = entry.get("@comunica/persistent-cache-manager:sourceState") or \
                        entry.get("@comunica/persistent-cache-manager:sourceStateQuerySource")

            hit_rate = np.nan
            if cache_str:
                try:
                    cache_stats = json.loads(cache_str)
                    hits = cache_stats.get("hits", 0)
                    total = hits + cache_stats.get("misses", 0)
                    hit_rate = (hits / total) if total > 0 else 0.0
                except json.JSONDecodeError:
                    pass

            # Determine session state
            if prev_session_id is None or (
                    current_session_id not in seen_sessions and current_session_id != prev_session_id):
                switch_type = "new_session"
            elif current_session_id == prev_session_id:
                switch_type = "within_session"
            else:
                switch_type = "existing_session"

            if template and not np.isnan(hit_rate):
                records.append({
                    'algorithm': algo_label,
                    'template': template,
                    'switch_type': switch_type,
                    'hit_rate': hit_rate
                })

            seen_sessions.add(current_session_id)
            prev_session_id = current_session_id

    if not records:
        return pd.DataFrame()

    df = pd.DataFrame(records)

    # Calculate means and pivot
    avg_df = df.groupby(['algorithm', 'template', 'switch_type'])['hit_rate'].mean().reset_index()

    pivot_df = avg_df.pivot(
        index=['algorithm', 'template'],
        columns='switch_type',
        values='hit_rate'
    ).reset_index()

    pivot_df.columns.name = None

    # Standardize output columns
    expected_columns = ['new_session', 'existing_session', 'within_session']
    for col in expected_columns:
        if col not in pivot_df.columns:
            pivot_df[col] = np.nan

    col_order = ['algorithm', 'template', 'new_session', 'existing_session', 'within_session']
    pivot_df = pivot_df[[c for c in col_order if c in pivot_df.columns]]

    return pivot_df

calculate_session_hit_rates(files)

,algorithm,template,new_session,existing_session,within_session
0,cache-l,interactive-discover-1,1.000000,0.903148,0.622414
1,cache-l,interactive-discover-2,NaN,0.607878,0.635323
2,cache-l,interactive-discover-3,NaN,0.148294,0.438040
3,cache-l,interactive-discover-4,0.350575,0.484481,0.550039
4,cache-l,interactive-discover-5,NaN,0.673723,0.571177
...,...,...,...,...,...
130,query-cache-s,interactive-short-3,0.013158,0.031646,0.306410
131,query-cache-s,interactive-short-4,0.666667,0.587121,0.786168
132,query-cache-s,interactive-short-5,NaN,0.444638,0.985554
133,query-cache-s,interactive-short-6,0.074475,0.043110,0.020966


In [28]:
import json
import os
from typing import List, Tuple, Dict, Any
import jinja2

import numpy as np
import pandas as pd


def extract_cache_stats(entry: Dict[str, Any]) -> Tuple[int, int]:
    """Extracts hit and total counts from the cache state JSON string."""
    cache_str = entry.get("@comunica/persistent-cache-manager:sourceState") or \
                entry.get("@comunica/persistent-cache-manager:sourceStateQuerySource")

    if not cache_str:
        return 0, 0

    try:
        cache_stats = json.loads(cache_str)
        hits = cache_stats.get("hits", 0)
        total = hits + cache_stats.get("misses", 0)
        return hits, total
    except json.JSONDecodeError:
        return 0, 0


def calculate_switch_effect(filenames: List[str]) -> pd.DataFrame:
    """
    Parses result files, normalizes hit rates against arithmetic template baselines,
    and aggregates absolute and relative hit rate deltas of session states.
    """
    records = []

    for path in sorted(filenames):
        algo_label = os.path.basename(path).replace("query-results-raw-", "").replace(".json", "")

        with open(path, 'r') as file:
            data = json.load(file)

        seen_sessions = set()
        prev_session_id = None

        for entry in data:
            seq_element = entry.get("sequenceElement", {})
            session_info = seq_element.get("session", {})
            current_session_id = session_info.get("sessionId")

            if not current_session_id:
                continue

            template = seq_element.get("template")
            hits, total = extract_cache_stats(entry)

            if total == 0 or not template:
                seen_sessions.add(current_session_id)
                prev_session_id = current_session_id
                continue

            if prev_session_id is None or (
                    current_session_id not in seen_sessions and current_session_id != prev_session_id):
                switch_type = "new_session"
            elif current_session_id == prev_session_id:
                switch_type = "within_session"
            else:
                switch_type = "existing_session"

            records.append({
                'algorithm': algo_label,
                'template': template,
                'switch_type': switch_type,
                'hit_rate': hits / total
            })

            seen_sessions.add(current_session_id)
            prev_session_id = current_session_id

    if not records:
        return pd.DataFrame()

    df = pd.DataFrame(records)

    # 1. Calculate baseline using arithmetic mean
    baselines = df.groupby(['algorithm', 'template'])['hit_rate'].mean().reset_index()
    baselines.rename(columns={'hit_rate': 'template_baseline'}, inplace=True)

    # 2. Merge baselines
    df = df.merge(baselines, on=['algorithm', 'template'])

    # 3. Aggregate rates by algorithm and switch type FIRST
    effect_df = df.groupby(['algorithm', 'switch_type'])[['hit_rate', 'template_baseline']].mean().reset_index()

    # 4. Calculate absolute and percentage deltas on the aggregated means
    effect_df['abs_delta'] = effect_df['hit_rate'] - effect_df['template_baseline']
    effect_df['pct_delta'] = np.where(
        effect_df['template_baseline'] == 0,
        np.nan,
        (effect_df['abs_delta'] / effect_df['template_baseline']) * 100
    )

    # 5. Pivot for presentation
    pivot_df = effect_df.pivot(
        index='algorithm',
        columns='switch_type',
        values=['abs_delta', 'pct_delta']
    )

    # Flatten the MultiIndex columns
    # Reverts absolute columns to original names (e.g., 'new_session')
    # and appends '_pct' for percentages (e.g., 'new_session_pct')
    new_cols = []
    for val, switch in pivot_df.columns:
        if val == 'abs_delta':
            new_cols.append(switch)
        else:
            new_cols.append(f"{switch}_pct")

    pivot_df.columns = new_cols
    pivot_df = pivot_df.reset_index()

    # Enforce column ordering, injecting NaNs for missing switch types
    base_columns = ['new_session', 'existing_session', 'within_session']
    col_order = ['algorithm']

    for base in base_columns:
        pct_col = f"{base}_pct"

        if base not in pivot_df.columns:
            pivot_df[base] = np.nan
        if pct_col not in pivot_df.columns:
            pivot_df[pct_col] = np.nan

        col_order.extend([base, pct_col])

    # Filter to the final structured order
    pivot_df = pivot_df[[c for c in col_order if c in pivot_df.columns]]

    return pivot_df

calculate_switch_effect(files)

# output_df = calculate_switch_effect(files)
# output_df.to_csv(float_format='%.3f')

,algorithm,new_session,new_session_pct,existing_session,existing_session_pct,within_session,within_session_pct
0,cache-l,-0.065599,-12.277512,-0.013696,-2.474097,0.011273,1.930847
1,cache-m,-0.056492,-15.649340,-0.030931,-7.900162,0.024277,5.485630
2,cache-s,-0.070885,-24.274723,-0.021268,-6.551276,0.017106,4.582255
3,query-cache-estimate-l,-0.053117,-10.573164,-0.014745,-2.796064,0.011924,2.127418
4,query-cache-estimate-m,0.028221,7.173648,-0.000269,-0.064370,-0.000160,-0.035155
5,query-cache-estimate-s,-0.043950,-14.305423,-0.012746,-3.921012,0.010270,2.924068
6,query-cache-l,-0.026489,-5.010850,-0.003197,-0.571290,0.002776,0.467111
7,query-cache-m,-0.086832,-22.781769,-0.021434,-5.097853,0.017457,3.682125
8,query-cache-s,-0.033353,-11.036464,-0.014856,-4.470083,0.011740,3.153413
